In [1]:
import pandas as pd

inventory = pd.read_csv(
    "../outputs/inventory_forecast.csv"
)

inventory["date"] = pd.to_datetime(inventory["date"])

inventory.head()

,date,site_id,consumed_tonnes,predicted_tonnes,cement_type,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,silo_capacity,inventory_change,starting_inventory,inventory_tonnes
0,2024-11-06,SITE_001,31.27,34.8780,CEM_III,6.10,25.17,0.00,448,-9.7080,6.1,-3.6080
1,2024-11-07,SITE_001,26.91,23.2376,CEM_I,0.00,41.23,14.32,448,17.9924,6.1,14.3844
2,2024-11-08,SITE_001,45.03,30.8429,CEM_III,14.32,48.88,18.17,448,18.0371,6.1,32.4215
3,2024-11-09,SITE_001,42.58,35.5245,CEM_III,18.17,36.63,12.22,448,1.1055,6.1,33.5270
4,2024-11-10,SITE_001,28.89,37.5600,CEM_I,12.22,16.67,0.00,448,-20.8900,6.1,12.6370


We want to create risk such as the Stockout risk: inventory is zero or negative.
Overcapacity risk: inventory is greater than the silo capacity.

In [2]:
# Detect projected stockouts
inventory["stockout_risk"] = (
    inventory["inventory_tonnes"] <= 0
)

# Detect projected overcapacity
inventory["overcapacity_risk"] = (
    inventory["inventory_tonnes"]
    > inventory["silo_capacity"]
)

# Count the risk days
inventory[
    ["stockout_risk", "overcapacity_risk"]
].sum()

stockout_risk        381
overcapacity_risk    718
dtype: int64

We will use a 3-day delivery lead time.

The reorder point means: “How much cement does a site need to survive the three days before a new delivery arrives?”

In [3]:
# Delivery lead time
lead_time_days = 3

# Average predicted daily demand for each site
inventory["average_daily_demand"] = (
    inventory.groupby("site_id")[
        "predicted_tonnes"
    ].transform("mean")
)

# Minimum inventory needed during the delivery wait
inventory["reorder_point_tonnes"] = (
    inventory["average_daily_demand"]
    * lead_time_days
)

# Low stock: above zero but below the reorder point
inventory["low_stock_risk"] = (
    (inventory["inventory_tonnes"] > 0)
    & (
        inventory["inventory_tonnes"]
        <= inventory["reorder_point_tonnes"]
    )
)

inventory["low_stock_risk"].sum()

np.int64(302)

Let now combine the three risk checks into one clear risk_status column for the dashboard.

In [4]:
# Start every record as Normal
inventory["risk_status"] = "Normal"

# Assign the relevant risk status
inventory.loc[
    inventory["low_stock_risk"],
    "risk_status"
] = "Low Stock"

inventory.loc[
    inventory["stockout_risk"],
    "risk_status"
] = "Stockout"

inventory.loc[
    inventory["overcapacity_risk"],
    "risk_status"
] = "Overcapacity"

# Count each status
inventory["risk_status"].value_counts()

risk_status
Overcapacity    718
Stockout        381
Low Stock       302
Normal          279
Name: count, dtype: int64

Now calculate the recommended order quantity. It will restore low or negative inventory to the 3-day reorder point without exceeding silo capacity.

In [5]:
# Target inventory cannot exceed silo capacity
inventory["target_inventory_tonnes"] = inventory[
    ["reorder_point_tonnes", "silo_capacity"]
].min(axis=1)

# Quantity required to reach the target
inventory["reorder_quantity_tonnes"] = (
    inventory["target_inventory_tonnes"]
    - inventory["inventory_tonnes"]
).clip(lower=0).round(2)

# View the recommendations
inventory[
    [
        "date",
        "site_id",
        "risk_status",
        "inventory_tonnes",
        "reorder_point_tonnes",
        "reorder_quantity_tonnes"
    ]
].head(10)

,date,site_id,risk_status,inventory_tonnes,reorder_point_tonnes,reorder_quantity_tonnes
0,2024-11-06,SITE_001,Stockout,-3.6080,82.835245,86.44
1,2024-11-07,SITE_001,Low Stock,14.3844,82.835245,68.45
2,2024-11-08,SITE_001,Low Stock,32.4215,82.835245,50.41
3,2024-11-09,SITE_001,Low Stock,33.5270,82.835245,49.31
4,2024-11-10,SITE_001,Low Stock,12.6370,82.835245,70.20
5,2024-11-11,SITE_001,Low Stock,6.5336,82.835245,76.30
6,2024-11-12,SITE_001,Low Stock,15.3752,82.835245,67.46
7,2024-11-13,SITE_001,Low Stock,32.9952,82.835245,49.84
8,2024-11-14,SITE_001,Low Stock,32.2889,82.835245,50.55
9,2024-11-15,SITE_001,Low Stock,15.1552,82.835245,67.68


In [6]:
#Now convert each risk into a clear business action:
# Default action
inventory["recommended_action"] = "No Action"

# Actions for each risk
inventory.loc[
    inventory["low_stock_risk"],
    "recommended_action"
] = "Place Reorder"

inventory.loc[
    inventory["stockout_risk"],
    "recommended_action"
] = "Emergency Reorder"

inventory.loc[
    inventory["overcapacity_risk"],
    "recommended_action"
] = "Pause Delivery"

# Count the recommended actions
inventory["recommended_action"].value_counts()

recommended_action
Pause Delivery       718
Emergency Reorder    381
Place Reorder        302
No Action            279
Name: count, dtype: int64

In [7]:
# Saving the reorder dataset which will be used for the Dashboard
inventory.to_csv(
    "../outputs/risk_reorder_recommendations.csv",
    index=False
)